In [22]:
import pandas as pd

In [23]:
rating = pd.read_parquet(
    path=r"./datasets/rating.parquet",
    engine="pyarrow",
    columns=["userId", "movieId", "rating"],
)
rating.head()

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


In [24]:
movie = pd.read_parquet(
    path=r"./datasets/movie.parquet",
    engine="pyarrow",
)
movie.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [25]:
data = pd.merge(left=rating, right=movie, on="movieId")

In [26]:
data.head()

,userId,movieId,rating,title,genres
0,1,2,3.5,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [27]:
data.shape

(20000263, 5)

In [28]:
movie_rating_count = (
    data.groupby(by="title")["rating"]
    .count()
    .reset_index()
    .rename(columns={"rating": "TotalRatingCount"})
)

In [29]:
movie_rating_count.head()

,title,TotalRatingCount
0,#chicagoGirl: The Social Network Takes on a Di...,3
1,$ (Dollars) (1971),24
2,$5 a Day (2008),39
3,$9.99 (2008),55
4,$ellebrity (Sellebrity) (2012),2


In [30]:
final = data.merge(right=movie_rating_count, on="title", how="left")

In [31]:
final.head()

,userId,movieId,rating,title,genres,TotalRatingCount
0,1,2,3.5,Jumanji (1995),Adventure|Children|Fantasy,22243
1,1,29,3.5,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,8520
2,1,32,3.5,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,44980
3,1,47,3.5,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,43249
4,1,50,3.5,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,47006


In [32]:
final.to_parquet(path=r"./datasets/clean_data.parquet", engine="pyarrow", index=False)